In [6]:
import numpy as np
import random
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

# 0 = open path, 1 = wall
maze = np.array([
    [0, 0, 0, 0, 0, 0],
    [1, 1, 0, 1, 1, 0],
    [0, 0, 0, 0, 1, 0],
    [0, 1, 1, 0, 1, 0],
    [0, 1, 0, 0, 0, 0],
    [0, 0, 0, 1, 1, 0]
])

start = (0, 0)
goal = (5, 5)
n_rows, n_cols = maze.shape

In [7]:
actions = ['up', 'down', 'left', 'right']

def move(state, action):
    r, c = state
    if action == 'up': r -= 1
    elif action == 'down': r += 1
    elif action == 'left': c -= 1
    elif action == 'right': c += 1

    if r < 0 or r >= n_rows or c < 0 or c >= n_cols or maze[r, c] == 1:
        return state
    return (r, c)

def get_reward(state):
    if state == goal:
        return 100
    return -1

In [8]:
Q = {}
for r in range(n_rows):
    for c in range(n_cols):
        for a in actions:
            Q[((r, c), a)] = 0.0

alpha = 0.1
gamma = 0.9
epsilon = 0.2
episodes = 2000

for episode in range(episodes):
    state = start
    steps = 0
    while state != goal and steps < 150:
        if random.random() < epsilon:
            action = random.choice(actions)
        else:
            q_vals = [Q[(state, a)] for a in actions]
            action = actions[np.argmax(q_vals)]

        next_state = move(state, action)
        reward = get_reward(next_state)
        best_next = max(Q[(next_state, a)] for a in actions)
        Q[(state, action)] += alpha * (reward + gamma * best_next - Q[(state, action)])

        state = next_state
        steps += 1

print("Training done!")

Training done!


In [9]:
def solve_maze():
    state = start
    path = [state]
    for _ in range(100):
        if state == goal:
            break
        q_vals = [Q[(state, a)] for a in actions]
        action = actions[np.argmax(q_vals)]
        state = move(state, action)
        path.append(state)
    return path

path = solve_maze()
print("Path:", path)
print("Reached goal!" if path[-1] == goal else "Did NOT reach goal — try increasing episodes")

Path: [(0, 0), (0, 1), (0, 2), (0, 3), (0, 4), (0, 5), (1, 5), (2, 5), (3, 5), (4, 5), (5, 5)]
Reached goal!


In [11]:
fig, ax = plt.subplots(figsize=(6, 6))

def draw_maze(ax):
    ax.clear()
    ax.imshow(maze, cmap='binary')
    ax.set_xticks(np.arange(-0.5, n_cols, 1), minor=True)
    ax.set_yticks(np.arange(-0.5, n_rows, 1), minor=True)
    ax.grid(which='minor', color='gray', linewidth=1)
    ax.set_xticks([])
    ax.set_yticks([])
    # Mark start and goal
    ax.scatter(start[1], start[0], color='green', s=200, marker='s', label='Start')
    ax.scatter(goal[1], goal[0], color='red', s=200, marker='*', label='Goal')

def update(frame):
    draw_maze(ax)
    r, c = path[frame]
    ax.scatter(c, r, color='blue', s=150, marker='o')  # agent position
    # draw trail so far
    trail = np.array(path[:frame+1])
    ax.plot(trail[:, 1], trail[:, 0], color='blue', alpha=0.4, linewidth=2)
    ax.set_title(f"Step {frame+1}/{len(path)}")

ani = FuncAnimation(fig, update, frames=len(path), interval=400, repeat=False)
plt.close()  # prevents duplicate static plot
HTML(ani.to_jshtml())